<a href="https://colab.research.google.com/github/2catch2/Predictive-Phase-Shift-Core-PPSC-/blob/main/PPSCDashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import math
import time
import random
import json
from http.server import HTTPServer, BaseHTTPRequestHandler
from threading import Thread

# =====================================================================
# 1. CORE PREDICTIVE PHASE-SHIFT CORE ENGINE (WITH P&L LOGIC)
# =====================================================================
class MorphicStateEngineV101:
    def __init__(self, variance_threshold=0.15, spike_threshold=0.85, initial_capital=100000.0):
        self.sigma = variance_threshold
        self.critical_max = spike_threshold
        self.current_state = "be"
        self.history_ledger = []
        self.mutation_counter = 0
        self.capital = initial_capital
        self.position = 0.0
        self.entry_price = 0.0
        self.total_profit_generated = 0.0

    def tool2_friction_harmonizer(self, normalized_price_change, rhythm_factor, volume_spike):
        if abs(volume_spike) >= self.critical_max or abs(normalized_price_change) >= self.critical_max:
            return "if"
        elif abs(rhythm_factor) >= 0.5:
            return "re"
        else:
            return "be"

    def process_market_token(self, token, timestamp, reference_price):
        log_entry = {
            "timestamp": time.strftime("%H:%M:%S", time.localtime(timestamp)),
            "token": token.upper(),
            "pre_state": self.current_state,
            "price": reference_price,
            "action": "EVALUATING",
            "capital_balance": round(self.capital, 2),
            "profit_delta": 0.0,
            "total_pnl": round(self.total_profit_generated, 2)
        }

        if self.current_state == "be" and token == "if":
            self.current_state = "be re be if if"
            log_entry["action"] = "SHIELD DEPLOYED: Front-running volatility anomaly."
            allocation = self.capital * 0.25
            self.position = allocation / reference_price
            self.entry_price = reference_price
            self.capital -= allocation

        elif self.current_state == "be re be if if" and token == "re":
            self.current_state = "be re be if if re"
            log_entry["action"] = "MARKET COLLISION: High-frequency friction loop holding."

        elif self.current_state == "be re be if if re":
            self.current_state = "be re be [if if] re"
            exit_price = reference_price * random.uniform(1.03, 1.06)
            gross_return = self.position * exit_price
            profit = gross_return - (self.position * self.entry_price)
            self.capital += gross_return
            self.total_profit_generated += profit
            self.position = 0.0
            self.current_state = "be*"
            self.mutation_counter += 1
            log_entry["action"] = f"MUTATION: Paradox archived. Profit +${round(profit, 2)}."
            log_entry["profit_delta"] = round(profit, 2)

        elif self.current_state == "be*" and token == "be":
            scalp_profit = random.uniform(25.0, 95.0)
            self.capital += scalp_profit
            self.total_profit_generated += scalp_profit
            log_entry["action"] = f"HYPER-BYPASS: Zero-latency scalp trade +${round(scalp_profit, 2)}."
            log_entry["profit_delta"] = round(scalp_profit, 2)
            if random.random() > 0.6:
                self.current_state = "be"

        log_entry["total_pnl"] = round(self.total_profit_generated, 2)
        log_entry["capital_balance"] = round(self.capital, 2)
        self.history_ledger.append(log_entry)
        return log_entry

class MarketDataIngester:
    def __init__(self):
        self.base_price = 85000.0
        self.tick_count = 0

    def get_next_tick(self):
        self.tick_count += 1
        t = self.tick_count
        rhythmic_loop = 0.6 * math.sin(2 * math.pi * 0.15 * t)
        ambient_noise = random.uniform(-0.1, 0.1)
        anomaly_spike = 0.0
        if t % 6 == 0:
            anomaly_spike = random.uniform(0.9, 1.3)
        elif (t - 1) % 6 == 0 and t > 1:
            rhythmic_loop = 0.85
        total_delta = ambient_noise + rhythmic_loop + anomaly_spike
        self.base_price += (total_delta * 140.0)
        return {
            "timestamp": time.time(),
            "price": round(self.base_price, 2),
            "normalized_change": total_delta,
            "rhythm_factor": rhythmic_loop,
            "volume_spike": anomaly_spike
        }

# Global instances
engine = MorphicStateEngineV101()
ingester = MarketDataIngester()

# Pre-populate engine with initial data ticks to give chart historical background immediately
for _ in range(15):
    td = ingester.get_next_tick()
    tk = engine.tool2_friction_harmonizer(td["normalized_change"], td["rhythm_factor"], td["volume_spike"])
    engine.process_market_token(tk, td["timestamp"], td["price"])

# =====================================================================
# 2. HIGH-SPEED INTEGRATED WEB SERVER & UI RENDERING
# =====================================================================
class DashboardServerHandler(BaseHTTPRequestHandler):
    def log_message(self, format, *args):
        return # Suppress default logging to keep terminal completely clean

    def _set_headers(self, content_type="application/json"):
        self.send_response(200)
        self.send_header("Content-Type", content_type)
        self.send_header("Access-Control-Allow-Origin", "*")
        self.end_headers()

    def do_GET(self):
        # API Endpoint: Live Stream Feed Data
        if self.path == "/api/stream":
            self._set_headers("application/json")
            td = ingester.get_next_tick()
            tk = engine.tool2_friction_harmonizer(td["normalized_change"], td["rhythm_factor"], td["volume_spike"])
            res = engine.process_market_token(tk, td["timestamp"], td["price"])

            payload = {
                "metrics": {
                    "state": engine.current_state.upper(),
                    "mutations": engine.mutation_counter,
                    "capital": round(engine.capital, 2),
                    "net_pnl": round(engine.total_profit_generated, 2)
                },
                "latest_log": res,
                "history": engine.history_ledger[-30:] # Send trailing historical array
            }
            self.wfile.write(json.dumps(payload).encode("utf-8"))

        # API Endpoint: Export Trade History as CSV
        elif self.path == "/api/export_csv":
            self._set_headers("text/csv")
            self.send_header('Content-Disposition', 'attachment; filename="trade_history.csv"')
            self.end_headers()

            if not engine.history_ledger:
                self.wfile.write(b'No trade history available.')
                return

            # Write CSV header
            header = ",".join(engine.history_ledger[0].keys()) + "\n"
            self.wfile.write(header.encode("utf-8"))

            # Write CSV rows
            for entry in engine.history_ledger:
                row = ",".join(str(entry[key]) for key in entry.keys()) + "\n"
                self.wfile.write(row.encode("utf-8"))

        # UI Endpoint: Minimalist Dark Dashboard Interface
        elif self.path == "/" or self.path == "/index.html":
            self._set_headers("text/html")
            html_content = """
            <!DOCTYPE html>
            <html lang="en">
            <head>
                <meta charset="UTF-8">
                <title>PPSC V1.01 Evolution Engine Dashboard</title>
                <script src="https://jsdelivr.net"></script>
                <style>
                    body { background-color: #0d0f12; color: #e2e8f0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 0; padding: 24px; }
                    .container { max-width: 1200px; margin: 0 auto; }
                    .header { display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #1e293b; padding-bottom: 16px; margin-bottom: 24px; }
                    .title h1 { margin: 0; font-size: 24px; font-weight: 700; letter-spacing: -0.05em; color: #38bdf8; }
                    .title p { margin: 4px 0 0 0; color: #64748b; font-size: 14px; }
                    .grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 24px; }
                    .card { background-color: #141820; border: 1px solid #1e293b; padding: 20px; border-radius: 8px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.5); }
                    .card-label { font-size: 12px; text-transform: uppercase; color: #64748b; letter-spacing: 0.05em; font-weight: 600; }
                    .card-value { font-size: 28px; font-weight: 700; margin-top: 8px; font-family: monospace; }
                    .state-badge { display: inline-block; padding: 4px 12px; border-radius: 12px; font-size: 14px; font-weight: bold; background: #0369a1; color: #e0f2fe; }
                    .chart-container { background-color: #141820; border: 1px solid #1e293b; padding: 20px; border-radius: 8px; margin-bottom: 24px; height: 360px; }
                    .log-container { background-color: #141820; border: 1px solid #1e293b; border-radius: 8px; padding: 16px; height: 250px; overflow-y: auto; font-family: monospace; font-size: 13px; }
                    .log-row { padding: 8px 12px; border-bottom: 1px solid #1e293b; display: flex; justify-content: space-between; }
                    .log-row:last-child { border-bottom: none; }
                    .token-if { color: #f43f5e; font-weight: bold; }
                    .token-re { color: #fbbf24; font-weight: bold; }
                    .token-be { color: #10b981; font-weight: bold; }
                    .btn { background-color: #38bdf8; color: #0d0f12; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; display: inline-block; }
                    .btn:hover { background-color: #22d3ee; }
                </style>
            </head>
            <body>
                <div class="container">
                    <div class="header">
                        <div class="title">
                            <h1>Predictive Phase-Shift Core (PPSC)</h1>
                            <p>Morphic State Engine V1.01 Performance Framework</p>
                        </div>
                        <div>
                            <span id="engine-state-badge" class="state-badge">BE</span>
                            <a href="/api/export_csv" class="btn" style="margin-left: 10px;">Download CSV</a>
                        </div>
                    </div>

                    <div class="grid">
                        <div class="card"><div class="card-label">Net Profit (P&L)</div><div id="val-pnl" class="card-value" style="color:#10b981;">$0.00</div></div>
                        <div class="card"><div class="card-label">Total Capital</div><div id="val-capital" class="card-value">$100,000.00</div></div>
                        <div class="card"><div class="card-label">Mutations</div><div id="val-mutations" class="card-value">0</div></div>
                        <div class="card"><div class="card-label">Current State</div><div id="val-state" class="card-value"><span id="engine-state-text">BE</span></div></div>
                    </div> <!-- Close grid -->

                    <div class="chart-container">
                        <canvas id="priceChart"></canvas>
                    </div>

                    <div class="log-container" id="log-feed">
                        <!-- Log entries will be inserted here by JavaScript -->
                    </div>
                </div> <!-- Close container -->

                <script src="https://cdn.jsdelivr.net/npm/chart.js@3.7.0/dist/chart.min.js"></script>
                <script>
                    const pnlElement = document.getElementById('val-pnl');
                    const capitalElement = document.getElementById('val-capital');
                    const mutationsElement = document.getElementById('val-mutations');
                    const stateBadge = document.getElementById('engine-state-badge');
                    const stateText = document.getElementById('engine-state-text');
                    const logFeed = document.getElementById('log-feed');
                    const priceChartCanvas = document.getElementById('priceChart').getContext('2d');

                    const chart = new Chart(priceChartCanvas, {
                        type: 'line',
                        data: {
                            labels: [], // Timestamps
                            datasets: [{
                                label: 'Price',
                                data: [], // Price values
                                borderColor: '#38bdf8',
                                backgroundColor: 'rgba(56, 189, 248, 0.1)',
                                borderWidth: 2,
                                pointRadius: 0,
                                fill: true
                            }]
                        },
                        options: {
                            responsive: true,
                            maintainAspectRatio: false,
                            scales: {
                                x: {
                                    type: 'category',
                                    labels: [],
                                    ticks: {
                                        color: '#94a3b8'
                                    },
                                    grid: {
                                        color: '#1e293b'
                                    }
                                },
                                y: {
                                    ticks: {
                                        color: '#94a3b8'
                                    },
                                    grid: {
                                        color: '#1e293b'
                                    }
                                }
                            },
                            plugins: {
                                legend: {
                                    display: false
                                }
                            }
                        }
                    });

                    function updateDashboard() {
                        fetch('/api/stream')
                            .then(response => response.json())
                            .then(data => {
                                // Update metrics
                                pnlElement.textContent = '$' + data.metrics.net_pnl.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
                                capitalElement.textContent = '$' + data.metrics.capital.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
                                mutationsElement.textContent = data.metrics.mutations;
                                stateBadge.textContent = data.metrics.state;
                                stateText.textContent = data.metrics.state;

                                // Update chart
                                chart.data.labels = data.history.map(item => item.timestamp);
                                chart.data.datasets[0].data = data.history.map(item => item.price);
                                chart.update();

                                // Update logs
                                logFeed.innerHTML = ''; // Clear previous logs
                                data.history.forEach(log => {
                                    const logRow = document.createElement('div');
                                    logRow.className = 'log-row';
                                    const tokenClass = `token-${log.token.toLowerCase()}`;
                                    logRow.innerHTML = `
                                        <span>${log.timestamp}</span>
                                        <span>State: ${log.pre_state} -> ${log.current_state || data.metrics.state}</span>
                                        <span>Token: <span class="${tokenClass}">${log.token}</span></span>
                                        <span>Price: $${log.price.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>
                                        <span>Action: ${log.action}</span>
                                        <span>Capital: $${log.capital_balance.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>
                                        ${log.profit_delta !== 0 ? `<span style="color:#10b981;">P&L: +$${log.profit_delta.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>` : ''}
                                    `;
                                    logFeed.prepend(logRow); // Add to top
                                });

                                if (logFeed.children.length > 50) { // Keep log feed from getting too long
                                    logFeed.removeChild(logFeed.lastChild);
                                }
                            })
                            .catch(error => console.error('Error fetching data:', error));
                    }

                    // Update every 1 second
                    setInterval(updateDashboard, 1000);

                    // Initial load
                    updateDashboard();
                </script>
            </body>
            </html>
            """
            self.wfile.write(html_content.encode("utf-8"))

In [8]:
html_content = """
<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <title>PPSC V1.01 Evolution Engine Dashboard</title>
    <script src="https://jsdelivr.net"></script>
    <style>
        body { background-color: #0d0f12; color: #e2e8f0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, sans-serif; margin: 0; padding: 24px; }
        .container { max-width: 1200px; margin: 0 auto; }
        .header { display: flex; justify-content: space-between; align-items: center; border-bottom: 1px solid #1e293b; padding-bottom: 16px; margin-bottom: 24px; }
        .title h1 { margin: 0; font-size: 24px; font-weight: 700; letter-spacing: -0.05em; color: #38bdf8; }
        .title p { margin: 4px 0 0 0; color: #64748b; font-size: 14px; }
        .grid { display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; margin-bottom: 24px; }
        .card { background-color: #141820; border: 1px solid #1e293b; padding: 20px; border-radius: 8px; box-shadow: 0 4px 6px -1px rgba(0,0,0,0.5); }
        .card-label { font-size: 12px; text-transform: uppercase; color: #64748b; letter-spacing: 0.05em; font-weight: 600; }
        .card-value { font-size: 28px; font-weight: 700; margin-top: 8px; font-family: monospace; }
        .state-badge { display: inline-block; padding: 4px 12px; border-radius: 12px; font-size: 14px; font-weight: bold; background: #0369a1; color: #e0f2fe; }
        .chart-container { background-color: #141820; border: 1px solid #1e293b; padding: 20px; border-radius: 8px; margin-bottom: 24px; height: 360px; }
        .log-container { background-color: #141820; border: 1px solid #1e293b; border-radius: 8px; padding: 16px; height: 250px; overflow-y: auto; font-family: monospace; font-size: 13px; }
        .log-row { padding: 8px 12px; border-bottom: 1px solid #1e293b; display: flex; justify-content: space-between; }
        .log-row:last-child { border-bottom: none; }
        .token-if { color: #f43f5e; font-weight: bold; }
        .token-re { color: #fbbf24; font-weight: bold; }
        .token-be { color: #10b981; font-weight: bold; }
        .btn { background-color: #38bdf8; color: #0d0f12; padding: 8px 16px; border-radius: 6px; text-decoration: none; font-weight: bold; display: inline-block; }
        .btn:hover { background-color: #22d3ee; }
    </style>
</head>
<body>
    <div class="container">
        <div class="header">
            <div class="title">
                <h1>Predictive Phase-Shift Core (PPSC)</h1>
                <p>Morphic State Engine V1.01 Performance Framework</p>
            </div>
            <div>
                <span id="engine-state-badge" class="state-badge">BE</span>
                <a href="/api/export_csv" class="btn" style="margin-left: 10px;">Download CSV</a>
            </div>
        </div>

        <div class="grid">
            <div class="card"><div class="card-label">Net Profit (P&L)</div><div id="val-pnl" class="card-value" style="color:#10b981;">$0.00</div></div>
            <div class="card"><div class="card-label">Total Capital</div><div id="val-capital" class="card-value">$100,000.00</div></div>
            <div class="card"><div class="card-label">Mutations</div><div id="val-mutations" class="card-value">0</div></div>
            <div class="card"><div class="card-label">Current State</div><div id="val-state" class="card-value"><span id="engine-state-text">BE</span></div></div>
        </div> <!-- Close grid -->

        <div class="chart-container">
            <canvas id="priceChart"></canvas>
        </div>

        <div class="log-container" id="log-feed">
            <!-- Log entries will be inserted here by JavaScript -->
        </div>
    </div> <!-- Close container -->

    <script src="https://cdn.jsdelivr.net/npm/chart.js@3.7.0/dist/chart.min.js"></script>
    <script>
        const pnlElement = document.getElementById('val-pnl');
        const capitalElement = document.getElementById('val-capital');
        const mutationsElement = document.getElementById('val-mutations');
        const stateBadge = document.getElementById('engine-state-badge');
        const stateText = document.getElementById('engine-state-text');
        const logFeed = document.getElementById('log-feed');
        const priceChartCanvas = document.getElementById('priceChart').getContext('2d');

        const chart = new Chart(priceChartCanvas, {
            type: 'line',
            data: {
                labels: [], // Timestamps
                datasets: [{
                    label: 'Price',
                    data: [], // Price values
                    borderColor: '#38bdf8',
                    backgroundColor: 'rgba(56, 189, 248, 0.1)',
                    borderWidth: 2,
                    pointRadius: 0,
                    fill: true
                }]
            },
            options: {
                responsive: true,
                maintainAspectRatio: false,
                scales: {
                    x: {
                        type: 'category',
                        labels: [],
                        ticks: {
                            color: '#94a3b8'
                        },
                        grid: {
                            color: '#1e293b'
                        }
                    },
                    y: {
                        ticks: {
                            color: '#94a3b8'
                        },
                        grid: {
                            color: '#1e293b'
                        }
                    }
                },
                plugins: {
                    legend: {
                        display: false
                    }
                }
            }
        });

        function updateDashboard() {
            // Note: This fetch will not work when run offline as it relies on the Colab server.
            // For offline use, the data would need to be embedded or loaded from a local source.
            fetch('/api/stream')
                .then(response => response.json())
                .then(data => {
                    // Update metrics
                    pnlElement.textContent = '$' + data.metrics.net_pnl.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
                    capitalElement.textContent = '$' + data.metrics.capital.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2});
                    mutationsElement.textContent = data.metrics.mutations;
                    stateBadge.textContent = data.metrics.state;
                    stateText.textContent = data.metrics.state;

                    // Update chart
                    chart.data.labels = data.history.map(item => item.timestamp);
                    chart.data.datasets[0].data = data.history.map(item => item.price);
                    chart.update();

                    // Update logs
                    logFeed.innerHTML = ''; // Clear previous logs
                    data.history.forEach(log => {
                        const logRow = document.createElement('div');
                        logRow.className = 'log-row';
                        const tokenClass = `token-${log.token.toLowerCase()}`;
                        logRow.innerHTML = `
                            <span>${log.timestamp}</span>
                            <span>State: ${log.pre_state} -> ${log.current_state || data.metrics.state}</span>
                            <span>Token: <span class="${tokenClass}">${log.token}</span></span>
                            <span>Price: $${log.price.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>
                            <span>Action: ${log.action}</span>
                            <span>Capital: $${log.capital_balance.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>
                            ${log.profit_delta !== 0 ? `<span style="color:#10b981;">P&L: +$${log.profit_delta.toLocaleString(undefined, {minimumFractionDigits: 2, maximumFractionDigits: 2})}</span>` : ''}
                        `;
                        logFeed.prepend(logRow); // Add to top
                    });

                    if (logFeed.children.length > 50) { // Keep log feed from getting too long
                        logFeed.removeChild(logFeed.lastChild);
                    }
                })
                .catch(error => console.error('Error fetching data:', error));
        }

        // Update every 1 second
        // This will not update when run offline as it relies on the Colab server for data.
        setInterval(updateDashboard, 1000);

        // Initial load
        updateDashboard();
    </script>
</body>
</html>
"""

with open('dashboard.html', 'w') as f:
    f.write(html_content)

print("dashboard.html has been created. You can download it from the file browser on the left.")

dashboard.html has been created. You can download it from the file browser on the left.


In [3]:
PORT = 8000

def run_server():
    server_address = ('', PORT)
    httpd = HTTPServer(server_address, DashboardServerHandler)
    print(f'Starting httpd server on port {PORT}...')
    httpd.serve_forever()

# Start the server in a separate thread
server_thread = Thread(target=run_server)
server_thread.daemon = True # Allow the main program to exit even if the thread is still running
server_thread.start()

print(f'Server running in background. Access the dashboard at http://localhost:{PORT}')

Server running in background. Access the dashboard at http://localhost:8000
Starting httpd server on port 8000...
